# Heuers' statistical Football model

The following implementation aims to recreate the Football model of Prof. Dr. Andreas Heuer.

In [221]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from functools import lru_cache
import warnings
from sklearn.linear_model import LinearRegression

NUM_OF_COUNTRIES = 5
NUM_OF_SEASONS = 25
NUM_OF_MATCHES = 9130
NUM_OF_MATCHES_IN_A_SEASON = 1826
SEED = 1893

np.random.seed(SEED)
warnings.filterwarnings('ignore')


In [222]:
# Helperfunctions
def score_from_amount_and_diff(amount, diff):
    if diff < 0:
        home_score = (amount + diff) / 2
        away_score = home_score - diff
    else:
        away_score = (amount - diff) / 2
        home_score = away_score + diff

    # cut excess floating numbers
    if (home_score - int(home_score)) == 0 and (away_score - int(away_score)) == 0:
        home_score = int(home_score)
        away_score = int(away_score)
    
    return home_score, away_score


def print_formatted_pred_score(country, season, matchnumber, ht_name, at_name, ht_pred_score, at_pred_score, ht_actual_score, at_actual_score):
    print("%4s %3s %3s %20s %20s %15s | %6s" %
          (country, season, matchnumber, ht_name, at_name,
           str(ht_pred_score)[0:4] + ":" + str(at_pred_score)[0:4],
           f"{ht_actual_score}:{at_actual_score}"))


def closed_form_solution(X, y):
    # alternative: np.dot(np.linalg.inv(np.dot(X.T, X)), np.dot(X.T, y))
    return np.linalg.solve(np.dot(X.T, X), np.dot(X.T, y))


## Step 1 - Homefieldadvantage (per country and season)

In [223]:
df_homefield = pd.read_csv('data/plot_homefield_adv.csv', sep=';')
avg_homefield_adv = np.sum(df_homefield["homefield_adv"]) / NUM_OF_SEASONS
print("Average homefield advantage on all games:", avg_homefield_adv)


Average homefield advantage on all games: 0.36837358101135187


In [224]:
df_country_season_homefield_adv = pd.read_csv(
    "data/country_season_homefield_adv.csv", sep=";")

df_country_season_homefield_adv

,country,season,matchnumber,hf_adv_last_three_seasons,hf_adv_current_season,hf_adv_actual
0,ENG,14,1,0.000000,-0.300000,-0.6
1,ENG,14,2,0.000000,0.033333,0.7
2,ENG,14,3,0.000000,-0.075000,-0.4
3,ENG,14,4,0.000000,0.060000,0.6
4,ENG,14,5,0.000000,0.016667,-0.2
...,...,...,...,...,...,...
925,ITA,18,34,0.317094,0.262857,0.5
926,ITA,18,35,0.317094,0.247222,-0.3
927,ITA,18,36,0.317094,0.248649,0.3
928,ITA,18,37,0.317094,0.268421,1.0


In [243]:
@lru_cache(maxsize=None)
def h_country_season_homefield_adv(country, season, matchnumber):
    if season == 14 and matchnumber == 1:
        return avg_homefield_adv
        
    sub_df = df_country_season_homefield_adv.query(
        f"country == '{country}' and season < {season} or (country == '{country}' and season == {season} and matchnumber < {matchnumber})")

    X = sub_df[["hf_adv_last_three_seasons",
                "hf_adv_current_season"]].values
    y = sub_df["hf_adv_actual"].values

    # w = closed_form_solution(X, y)
    model = LinearRegression(positive=True).fit(X, y)
    print("COEF | INTERCEPT",model.coef_, "|",model.intercept_)
    
    input_df = df_country_season_homefield_adv.query(
        f"country == '{country}' and season == {season} and matchnumber == {matchnumber}")
    input = input_df[["hf_adv_last_three_seasons",
                      "hf_adv_current_season"]].values

    # return np.dot(w, input.T)[0]
    return model.predict(input)[0]

print("Homefield advantage:", h_country_season_homefield_adv("ESP", 14, 4))

for i in range(1,35):
    print("MatchNo", i, h_country_season_homefield_adv("GER",18,i))


COEF | INTERCEPT [0. 0.] | 0.6333333333333334
Homefield advantage: 0.6333333333333334
COEF | INTERCEPT [0.         1.91787673] | -0.2675084626515559
MatchNo 1 0.4783324866385429
COEF | INTERCEPT [0.00676924 1.92714213] | -0.27045665422293896
MatchNo 2 -0.196545242676915
COEF | INTERCEPT [0.00201732 2.01207151] | -0.30166156125948795
MatchNo 3 0.20211201299118897
COEF | INTERCEPT [0.01535686 1.97666365] | -0.28841317020025864
MatchNo 4 0.11267228742535623
COEF | INTERCEPT [0.0134661  1.98567049] | -0.29175163313799
MatchNo 5 0.44872632655023126
COEF | INTERCEPT [0.03250258 2.00190579] | -0.29729756411660474
MatchNo 6 0.6681664735905626
COEF | INTERCEPT [0.04533252 2.03824696] | -0.31033765767401
MatchNo 7 0.697458542980971
COEF | INTERCEPT [0.0412502  2.02604957] | -0.30595623341404915
MatchNo 8 0.5849497423245711
COEF | INTERCEPT [0.02596184 1.99449971] | -0.29472261597233373
MatchNo 9 0.4241582252015279
COEF | INTERCEPT [0.00866714 1.98709603] | -0.29237460887888234
MatchNo 10 0.27287

## Step 2 - Goalamount per match (per country and season)

there are higher scored countrys as well as lower scoring ones
for now the goalamount will be modeled as a constant

In [226]:
df_goalamount = pd.read_csv('data/plot_goalamount.csv', sep=';')
avg_goalamount = np.sum(df_goalamount["goalamount"]) / NUM_OF_SEASONS
print("Average goalamount of all games:", avg_goalamount)


Average goalamount of all games: 2.724046095631235


In [227]:
df_country_season_goalamount = pd.read_csv(
    "data/country_season_goalamount.csv", sep=";")

df_country_season_goalamount

,country,season,matchnumber,ga_last_three_seasons,ga_current_season,ga_actual
0,ENG,14,1,0.000000,0.000000,2.6
1,ENG,14,2,0.000000,2.600000,2.3
2,ENG,14,3,0.000000,2.450000,3.2
3,ENG,14,4,0.000000,2.700000,3.0
4,ENG,14,5,0.000000,2.775000,3.2
...,...,...,...,...,...,...
925,ITA,18,34,2.735965,2.663636,2.3
926,ITA,18,35,2.735965,2.652941,2.9
927,ITA,18,36,2.735965,2.660000,3.1
928,ITA,18,37,2.735965,2.672222,2.8


In [244]:
@lru_cache(maxsize=None)
def h_country_season_goalamount(country, season, matchnumber):
    if season == 14 and matchnumber == 1:
        return avg_goalamount
        
    sub_df = df_country_season_goalamount.query(
        f"country == '{country}' and season < {season} or (season == {season} and matchnumber < {matchnumber})")

    X = sub_df[["ga_last_three_seasons",
                "ga_current_season"]].values  # no bias
    y = sub_df["ga_actual"].values

    # w = closed_form_solution(X, y)
    model = LinearRegression(positive=True).fit(X, y)
    # print("COEF | INTERCEPT",model.coef_, "-",model.intercept_)

    input_df = df_country_season_goalamount.query(
        f"country == '{country}' and season == {season} and matchnumber == {matchnumber}")
    input = input_df[["ga_last_three_seasons",
                    "ga_current_season"]].values

    # return np.dot(w, input.T)[0]
    return model.predict(input)[0]


print("Goalamount:", h_country_season_goalamount("GER", 18, 18))


Goalamount: 2.809943346321686


## Step 3: Teamstrength and expected goaldifference + Step 4: Specialty for promoters

In [229]:
df_matches = pd.read_csv('data/matches_heuer.csv', sep=';')

df_matches

,country,season,matchnumber,teamname,curr_eff_chancediff,past_chancediff_combined,curr_eff_chanceamount,past_chanceamount,curr_fitness,market_value_ln_relative,is_promoter,is_home,actual_goalamount,actual_goaldiff
0,ENG,14,1,Queens Park Rangers,0.000000,0.000000,0.000000,0.000000,0.000000,-0.392415,True,True,1,-1
1,ENG,14,1,Hull,0.000000,0.000000,0.000000,0.000000,0.000000,-0.348740,False,False,1,1
2,ENG,14,1,Newcastle,0.000000,0.000000,0.000000,0.000000,0.000000,-0.169272,False,True,2,-2
3,ENG,14,1,Man City,0.000000,0.000000,0.000000,0.000000,0.000000,1.002768,False,False,2,2
4,ENG,14,1,Burnley,0.000000,0.000000,0.000000,0.000000,0.000000,-1.140622,True,True,4,-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18255,ITA,18,38,Napoli,5.242267,5.293421,3.415385,8.877193,3.375719,1.070083,False,False,5,-1
18256,ITA,18,38,Inter,2.881676,2.380263,1.042860,6.754386,1.768259,1.024225,False,True,3,1
18257,ITA,18,38,Empoli,-3.462648,-3.058772,-0.805584,4.403509,2.088522,-0.738824,True,False,3,-1
18258,ITA,18,38,Cagliari,-3.170633,-1.972368,-1.823289,4.824561,-3.719535,-0.358838,False,True,3,-1


In [230]:
@lru_cache(maxsize=None)
def h_teamstrength(country, season, matchnumber, teamname):
    if season == 14 and matchnumber == 1:
        return 0

    sub_df = df_matches.query(
        f"season < {season} and matchnumber == {matchnumber} or (season == {season} and matchnumber < {matchnumber})")

    X = sub_df[["past_chancediff_combined",
                "curr_eff_chancediff", "curr_fitness", "market_value_ln_relative"]].values # no bias
    y = []

    # one team is only responsible for half of the goaldiff
    for index, row in sub_df.iterrows():
        goal_diff = row["actual_goaldiff"]
        if row["is_home"]:
            goal_diff -= avg_homefield_adv # h_country_season_hf_adv
        else:
            goal_diff += avg_homefield_adv  # h_country_season_hf_adv
        y.append(goal_diff/2)

    # w = closed_form_solution(X, y)
    model = LinearRegression(positive=True).fit(X, y)
    # print("COEF | INTERCEPT",model.coef_, "-",model.intercept_)

    input_df = df_matches.query(
        f"season == {season} and matchnumber == {matchnumber} and teamname == '{teamname}'")

    # Homefield advantage correction
    homefield_matches = df_matches.query(
        f"season == {season} and matchnumber < {matchnumber} and teamname == '{teamname}' and is_home").count()["is_home"]

    if matchnumber > 1:
        additional_homefield_matches = homefield_matches - \
            ((matchnumber - 1) - homefield_matches)
        if additional_homefield_matches > 0:
            input_df["curr_eff_chancediff"] -= (additional_homefield_matches * \
                h_country_season_homefield_adv(country, season, matchnumber) * 4) / (matchnumber - 1)

    # Step 4: Specialty for promoters
    if input_df.iloc[0]["is_promoter"]:
        input_df["past_chancediff_combined"] = input_df["past_chancediff_combined"] * \
            0.45 * model.coef_[1]
        input_df["curr_eff_chancediff"] = input_df["curr_eff_chancediff"] * 0.45 * model.coef_[1]

    input = input_df[["past_chancediff_combined",
                      "curr_eff_chancediff", "curr_fitness","market_value_ln_relative"]].values

    # teamstrength = np.dot(w, input.T)[0]
    teamstrength = model.predict(input)[0]

    if input_df.iloc[0]["is_promoter"]:
        teamstrength -= (13/34)

    return teamstrength

print("Teamstrength:", h_teamstrength("ENG", 16, 12, "Leicester"))


Teamstrength: -0.04095944011261097


## Step 5: Expected goalamount per match

In [231]:
@lru_cache(maxsize=None)
def h_goalamount_from_avg_per_match(country, season, matchnumber, teamname):
    if season == 14 and matchnumber == 1:
        return 0

    sub_df = df_matches.query(
        f"season < {season} and matchnumber == {matchnumber} or (season == {season} and matchnumber < {matchnumber})")

    X = sub_df[["past_chanceamount", "curr_eff_chanceamount"]].values 
    y = []
    for index, row in sub_df.iterrows():
        y.append(row["actual_goalamount"] - avg_goalamount) # - h_country_season_goalamount(row["country"], row["season"], row["matchnumber"]))

    # w = closed_form_solution(X, y)
    model = LinearRegression(positive=True).fit(X, y)
    # print("COEF | INTERCEPT",model.coef_, "-",model.intercept_)

    input_df = df_matches.query(
        f"season == {season} and matchnumber == {matchnumber} and teamname == '{teamname}'")
    input = input_df[["past_chanceamount", "curr_eff_chanceamount"]].values

    # return np.dot(w, input.T)[0]
    return model.predict(input)[0]

print("Goalamount from avg:", h_goalamount_from_avg_per_match("GER", 16, 19, "Bayern"))


Goalamount from avg: 0.4361184922711733


## Step 6: Predicted Goals

In [232]:
def predicted_goals(country, season, matchnumber, hometeam, awayteam):
    goalamount = h_goalamount_from_avg_per_match(
        country, season, matchnumber, hometeam) + h_goalamount_from_avg_per_match(country, season, matchnumber, awayteam) + h_country_season_goalamount(country, season, matchnumber)
    goaldiff = h_country_season_homefield_adv(country, season, matchnumber) + h_teamstrength(country, season, matchnumber, hometeam) - h_teamstrength(country, season, matchnumber, awayteam)
    return (goalamount, goaldiff)


## Step 7: Dependence on matchnumber

In [233]:
def matchnumber_goalamount_multiplier(country, matchnumber):
    # Uncertain
    if country == 'GER':
        if matchnumber < 18:
            return 1.02
        elif matchnumber < 31:
            return 0.95
        else:
            return 1.2
    else:
        if matchnumber < 19:
            return 1.02
        elif matchnumber < 35:
            return 0.95
        else:
            return 1.2


## Final: Full run on dataset to measure goalamount and goaldifference error

In [234]:
def predict(country, season, matchnumber, hometeam, awayteam):
    amount, diff = predicted_goals(country, season, matchnumber, hometeam, awayteam)

    home_score, away_score = score_from_amount_and_diff(amount, diff)

    home_score *= matchnumber_goalamount_multiplier(country, matchnumber)
    away_score *= matchnumber_goalamount_multiplier(country, matchnumber)

    return (home_score+ away_score), (home_score - away_score)


In [235]:
# For match in Dataset run prediction and measure error
do_a_whole_run = True

if not do_a_whole_run:
    print("Skipped testing run")

test_df = df_matches.query(f"season == 18")
ga_abs_error, gd_abs_error = 0, 0

if do_a_whole_run:
    for index, row in test_df.iterrows():
        if index % 2 == 0:
            home_score, away_score = score_from_amount_and_diff(
                row["actual_goalamount"], row["actual_goaldiff"])
            hometeam = row["teamname"]
            continue
        else:
            awayteam = row["teamname"]
            ga_pred, gd_pred = predict(
                row["country"], row["season"], row["matchnumber"], hometeam, awayteam)
            ga_abs_error += np.abs(ga_pred - row["actual_goalamount"])
            gd_abs_error += np.abs(gd_pred - row["actual_goaldiff"])
            h_score_pred, a_score_pred = score_from_amount_and_diff(ga_pred,gd_pred)
            print_formatted_pred_score(row["country"], row["season"], row["matchnumber"], hometeam, awayteam, h_score_pred ,a_score_pred, home_score, away_score)

    print("Absolute error goalamount",
          ga_abs_error / NUM_OF_MATCHES_IN_A_SEASON)
    print("Absolute error goaldiff", gd_abs_error / NUM_OF_MATCHES_IN_A_SEASON)


 ENG  18   1               Wolves              Everton       1.20:1.52 |    2:2
 ENG  18   1              Watford             Brighton       1.48:1.17 |    2:0
 ENG  18   1          Southampton              Burnley       1.58:1.14 |    0:0
 ENG  18   1            Newcastle            Tottenham       1.00:1.97 |    1:2
 ENG  18   1              Man Utd            Leicester       1.74:1.14 |    2:1
 ENG  18   1              Arsenal             Man City       1.39:1.80 |    0:2
 ENG  18   1         Huddersfield              Chelsea       0.86:1.96 |    0:3
 ENG  18   1               Fulham       Crystal Palace       1.25:1.41 |    0:2
 ENG  18   1          Bournemouth              Cardiff       1.61:1.08 |    2:0
 ENG  18   1            Liverpool             West Ham       2.04:0.95 |    4:0
 ENG  18   2            Tottenham               Fulham       2.03:0.25 |    3:1
 ENG  18   2              Everton          Southampton       1.64:0.64 |    2:1
 ENG  18   2             West Ham       